In [ ]:
# --- arranque en Colab (no hace nada fuera de Colab) -----------------------
# Colab arranca en /content con un entorno vacío: sin esta celda el
# `import environment` siguiente falla con ModuleNotFoundError. Generada por
# tools/py_to_notebook.py -- editala ahí, no acá.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/solidgoldmagickarp/proyecto_de_lenguaje_emergente.git"
    REPO_DIR = "proyecto_de_lenguaje_emergente"

    # El orden de las guardas importa: primero probar DÓNDE estamos, después qué hay en disco.
    # `isdir(REPO_DIR)` es relativo, así que chequearlo primero clonaría una copia
    # anidada en cualquier re-ejecución de esta celda dentro de la misma sesión (algo rutinario).
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )

    # El disco de Colab es EFÍMERO: cada checkpoint se pierde cuando el entorno de ejecución
    # se desconecta (~90 min de inactividad en el plan gratuito), y las Etapas 3, 4 y 5 necesitan
    # todas los checkpoints de la Etapa 2. Descomentá para guardarlos en Drive en su lugar.
    #
    # Notar `islink`, no `exists`: importar `environment` crea un directorio checkpoints/
    # REAL, y `exists` entonces saltearía el enlace en silencio y mandaría cada
    # checkpoint de vuelta al disco efímero.
    #
    # from google.colab import drive
    # drive.mount("/content/drive")
    # PERSISTENT = "/content/drive/MyDrive/proyecto_de_lenguaje_emergente/checkpoints"
    # os.makedirs(PERSISTENT, exist_ok=True)
    # if os.path.islink("checkpoints"):
    #     print("checkpoints ->", os.readlink("checkpoints"))
    # elif os.path.isdir("checkpoints"):
    #     print("ADVERTENCIA: checkpoints/ ya es un directorio real, así que NO está en Drive.")
    #     print("Mové su contenido a", PERSISTENT, ", borralo, y volvé a ejecutar esta celda.")
    # else:
    #     os.symlink(PERSISTENT, "checkpoints")
    #     print("checkpoints ->", PERSISTENT)

    print("Arranque en Colab OK. Directorio de trabajo:", os.getcwd())

# Etapa 0 · El dataset como instrumento

Este notebook es el punto de partida: preparación del entorno y una primera mirada a los datos. Todavía no hay modelado.

TinyStories (Eldan & Li, 2023, arXiv:2305.07759) son cuentos cortos restringidos a un vocabulario que un
chico de 3-4 años entiende. Esa restricción no es una limitación que haya que sortear -- es lo que convierte
al dataset en un instrumento de laboratorio: aísla la variable "datos" del ruido de la escala, de modo que un
modelo de <10M de parámetros puede producir inglés coherente en lugar de balbuceo.

In [ ]:
import environment  # noqa: F401  (efectos colaterales: caché de HF, bundle de certificados, ...)

environment.summary()

SMOKE_TEST = True  # True: una porción mínima, corre en segundos. False: una muestra de inspección más grande.
SEED = 1337

## Cargar los datasets

Dos datasets, ambos traídos del HuggingFace Hub, nunca incluidos dentro de este repositorio:

- `roneneldan/TinyStories` -- el corpus de preentrenamiento (Etapa 2). Un dataset Parquet estándar, se corta en
  porciones de forma barata.
- `roneneldan/TinyStories-Instruct` -- el corpus de instruction-tuning (Etapa 4). Cada registro
  antepone un SUBCONJUNTO aleatorio, en ORDEN aleatorio, de hasta cuatro tipos de instrucción (`Features:`,
  `Words:`, `Summary:`, `Random sentence:`) antes del cuento -- confirmado muestreando varios
  miles de registros, no asumido. **Trampa:** este es un dataset legacy con script de carga, almacenado
  con una LÍNEA por fila, no un ejemplo por fila, y `datasets` materializa el split completo de ~21,7M de
  líneas antes de poder cortar nada -- esperá un costo único de ~20-30s en la primera corrida (después queda
  cacheado, no es que se colgó). `data.load_instruct_records` reagrupa las líneas en registros completos por vos; mirá el
  docstring de ese módulo para entender por qué el mirror de la comunidad (`skeskinen/TinyStories-Instruct-hf`) es peor
  acá, no mejor -- directamente rompe en una ruta de clonado profunda de Windows.

In [ ]:
from data import load_instruct_records, load_tinystories

INSPECT_N = 1_000 if SMOKE_TEST else 20_000

stories = load_tinystories(limit=INSPECT_N)
print(stories)
print()
print("--- un cuento crudo ---")
print(stories[0]["text"])

In [ ]:
instruct_records = load_instruct_records(limit=200 if SMOKE_TEST else 2_000)
print(f"registros instruct parseados: {len(instruct_records)}")
print()
print("--- un registro instruct crudo ---")
print(instruct_records[0])

## Distribución de longitudes

Un proxy por cantidad de palabras alcanza acá (la Etapa 1 te va a dar un conteo exacto de tokens una vez que tengas un
tokenizador). Esto importa para la Etapa 2: te dice qué parte de un cuento típico cubre realmente el
`block_size` (largo del contexto).

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(s.split()) for s in stories["text"]]
print(f"cuentos inspeccionados   : {len(lengths)}")
print(f"palabras/cuento (media)  : {sum(lengths) / len(lengths):.1f}")
print(f"palabras/cuento min / max: {min(lengths)} / {max(lengths)}")

plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=40)
plt.xlabel("palabras por cuento")
plt.ylabel("cantidad")
plt.title(f"Distribución de longitudes de TinyStories (n={len(lengths)})")
plt.tight_layout()
plt.savefig("checkpoints/stage0_length_hist.png", dpi=100)
plt.show()

## Un tamaño aproximado del vocabulario

Separando por espacios y pasando a minúsculas -- no es un tokenizador de verdad (eso es la Etapa 1), apenas lo suficiente
para sostener la afirmación del "vocabulario acotado" con un número en lugar de una intuición.

In [ ]:
from collections import Counter

word_counts = Counter(w.lower().strip(".,!?;:\"'") for s in stories["text"] for w in s.split())
print(f"palabras únicas (aprox.) : {len(word_counts)}")
print(f"las 10 más comunes       : {word_counts.most_common(10)}")

## Para tu informe

Argumentá, con los números de arriba, por qué un vocabulario acotado aísla la variable "datos" del
ruido de la escala. ¿Qué esperarías que cambie si entrenaras la misma arquitectura sobre una porción de
texto web general con la misma *cantidad de tokens* que esta muestra? Lo que sigue: Etapa 1 (`01_tokenizer.py`) --
entrenar un tokenizador BPE sobre este corpus.